In [3]:
# @title
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import accuracy_score, r2_score
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torchvision.datasets
import torchvision.transforms as transforms
from torch.utils.data import WeightedRandomSampler
import random
import glob
from scipy.ndimage import gaussian_filter, zoom
import pickle

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# @title
def set_seed(seed):
    """
    Use this to set ALL the random seeds to a fixed value and take out any randomness from cuda kernels
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False  ##uses the inbuilt cudnn auto-tuner to find the fastest convolution algorithms. -
    torch.backends.cudnn.enabled   = False

    return True

device = 'cpu'
if torch.cuda.device_count() > 0 and torch.cuda.is_available():
    print("Cuda installed! Running on GPU!")
    device = 'cuda'
else:
    print("No GPU available!")

Cuda installed! Running on GPU!


In [6]:
import sys
sys.path.append('/content/drive/MyDrive/IRP/scripts')

from SegNet import SegNet, CombinedLoss, SN_train, SegNetDataset
from plot_predictions import visualise
from ModelCompare import train_fold, _region_paths

In [10]:
REGION_GROUPS = {
    'Brazil': ['Brazil'],
    'USA': ['USA'],
    'Karoo': ['Karoo'],
    'Russia': ['Russia', 'Russia2', 'Russia3'],
    'Australia': ['Australia']
}

In [13]:
model_info = {'model': SegNet, 'loss': CombinedLoss(), 'train_fn': SN_train, 'dataset': SegNetDataset}
epochs = 50

In [14]:
cv_results = {}
best_states = {}

for held_out in REGION_GROUPS:
    print(f"\n=== fold: holding out {held_out} ===")
    val_paths = _region_paths(held_out)
    print(f'Found {len(val_paths)} for {held_out}')
    train_paths = [p for r in REGION_GROUPS if r != held_out for p in _region_paths(r)]

    metrics, best_model, _valloader = train_fold(train_paths, val_paths, epochs, model_info)
    cv_results[held_out] = metrics # Store metrics per held_out region



=== fold: holding out Brazil ===
Found 66 for Brazil
Found 187 tiles (0 partial, 0 missing bands skipped)
Found 66 tiles (0 partial, 0 missing bands skipped)
Best checkpoint saved (iou 0.2676)
Epoch   1/50 | train 0.7731 | val 0.7001 | acc 97.01% | iou 0.2676 | P 0.4089 | R 0.4363 | F1 0.4222 | Obj_P 0.1199 | Obj_R 0.6821 | Obj_F1 0.2040 | 
Epoch   2/50 | train 0.6773 | val 0.6715 | acc 97.33% | iou 0.2623 | P 0.4613 | R 0.3782 | F1 0.4156 | Obj_P 0.1634 | Obj_R 0.6243 | Obj_F1 0.2590 | 
Best checkpoint saved (iou 0.3458)
Epoch   3/50 | train 0.6284 | val 0.6020 | acc 97.39% | iou 0.3458 | P 0.4826 | R 0.5495 | F1 0.5139 | Obj_P 0.2042 | Obj_R 0.7225 | Obj_F1 0.3185 | 
Epoch   4/50 | train 0.6038 | val 0.6484 | acc 95.05% | iou 0.2190 | P 0.2660 | R 0.5534 | F1 0.3593 | Obj_P 0.1159 | Obj_R 0.6994 | Obj_F1 0.1988 | 
Epoch   5/50 | train 0.5998 | val 0.5892 | acc 97.18% | iou 0.3327 | P 0.4497 | R 0.5610 | F1 0.4992 | Obj_P 0.3033 | Obj_R 0.6821 | Obj_F1 0.4199 | 
Best checkpoint saved

KeyboardInterrupt: 

In [15]:
with open('/content/drive/MyDrive/IRP/results/seg100epochs_brazil_folded_out_metrics.pkl', 'wb') as f:
    pickle.dump(cv_results, f)

torch.save(best_model, '/content/drive/MyDrive/IRP/models/seg100_epochs_brazil_folded_out_model.pkl')
